# Code Interpreter: Python in an OpenAI Sandbox

Use the Responses API to analyse a CSV, generate a chart, and download verified results from a hosted Python sandbox.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/code_interpreter.ipynb) · [Official guide](https://developers.openai.com/api/docs/guides/tools-code-interpreter)

**The idea:** you describe the analysis; the model writes and runs Python on OpenAI's infrastructure. Python in *this notebook* sends the request and downloads the files.

**Filming path (about 4 minutes):** explain the sandbox, run the supplied setup, show the CSV, highlight `code_interpreter`, then inspect the chart and totals. Pre-run package installation before recording.

You need an OpenAI API key with billing and a model supporting Code Interpreter. API and container usage are billed separately from ChatGPT subscriptions. All data below is synthetic.

In [ ]:
%pip install openai --upgrade --quiet

In [ ]:
RUN_DEMO = True
CLEANUP = True  # Delete this demo's uploaded file and container after downloading.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Markdown, FileLink, Image

if not isinstance(RUN_DEMO, bool):
    raise TypeError("RUN_DEMO must be True or False, not text.")
if RUN_DEMO and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=180, max_retries=1) if RUN_DEMO else None
MODEL = os.getenv("OPENAI_CODE_INTERPRETER_MODEL", "gpt-5.4")
print("Model:", MODEL)

## 1. A small dataset we can check ourselves

We want **revenue by region**, a bar chart, and a downloadable summary CSV. Start with six rows so we can verify the answer without trusting the model's prose.

In [ ]:
import csv

workdir = Path("code_interpreter_demo")
workdir.mkdir(exist_ok=True)
sales_path = workdir / "sales.csv"
sales_path.write_text(
    "month,region,revenue\n"
    "Jan,North,120\nJan,South,100\n"
    "Feb,North,140\nFeb,South,120\n"
    "Mar,North,160\nMar,South,140\n",
    encoding="utf-8",
)
expected_totals = {}
with sales_path.open() as f:
    for row in csv.DictReader(f):
        region = row["region"]
        expected_totals[region] = expected_totals.get(region, 0) + int(row["revenue"])
print(sales_path.read_text())
print("Our check:", expected_totals, "Grand total:", sum(expected_totals.values()))

## 2. Upload the file and enable the Python tool

The important part is the **tools configuration**. `container.type = "auto"` creates or reuses an active sandbox; `file_ids` makes our uploaded CSV available to Python. We explicitly ask the model to **use the python tool**, rather than calculate from text alone.

In [ ]:
uploaded = None
response = None
container_ids = set()
if RUN_DEMO:
    with sales_path.open("rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")
    try:
        response = client.responses.create(
            model=MODEL,
            tools=[{"type": "code_interpreter",
                    "container": {"type": "auto", "file_ids": [uploaded.id]}}],
            instructions="Use the python tool to analyse the supplied CSV. Report actual calculated results.",
            input=("Read sales.csv. Calculate total revenue by region and the grand total. "
                   "Create regional_totals.csv with exactly the columns region,total_revenue. "
                   "Create revenue_by_region.png, a labelled bar chart. "
                   "Link both downloadable files in your final answer."),
            max_output_tokens=2500,
        )
    except Exception:
        client.files.delete(uploaded.id)
        uploaded = None
        raise
    if response.status != "completed":
        raise RuntimeError(f"Response stopped with {response.status}; inspect response.incomplete_details.")
    display(Markdown(response.output_text))

## 3. Inspect what actually ran

An answer is stronger when we can see that the tool executed. `code_interpreter_call` contains the container ID and generated Python. The outputs may include execution logs.

In [ ]:
if RUN_DEMO:
    calls = [item for item in response.output if item.type == "code_interpreter_call"]
    if not calls:
        raise RuntimeError("No Python tool call was returned. Check the model and prompt.")
    for call in calls:
        container_ids.add(call.container_id)
        print("Python execution:", call.status)
        print(call.code)
        for output in call.outputs or []:
            if output.type == "logs":
                print(output.logs)

## 4. Download the generated files and check the numbers

Generated-file links appear as `container_file_citation` annotations. Use the **container file API**, rather than the uploaded-file API, to download them. This saves the outputs locally so they survive container expiration.

In [ ]:
downloaded = {}
if RUN_DEMO:
    for item in response.output:
        if item.type != "message":
            continue
        for part in item.content:
            for citation in getattr(part, "annotations", []):
                if citation.type != "container_file_citation":
                    continue
                # Strip any directory components from the model-provided filename.
                filename = Path(citation.filename).name
                destination = workdir / filename
                content = client.containers.files.content.retrieve(
                    citation.file_id, container_id=citation.container_id,
                )
                content.write_to_file(destination)
                downloaded[filename] = destination
                container_ids.add(citation.container_id)
                display(FileLink(str(destination)))

    summary_path = downloaded.get("regional_totals.csv")
    chart_path = downloaded.get("revenue_by_region.png")
    if summary_path is None or chart_path is None:
        raise RuntimeError("Expected CSV and PNG were not both cited. Inspect the output and rerun with clearer filenames.")
    with summary_path.open() as f:
        actual = {row["region"]: float(row["total_revenue"]) for row in csv.DictReader(f)}
    assert actual == expected_totals, f"Calculation mismatch: {actual}"
    print("Verified totals:", actual, "Grand total:", sum(actual.values()))
    display(Image(filename=str(chart_path)))

## 5. Clean up the remote demo resources

The downloaded CSV and chart remain in `code_interpreter_demo/`. Colab's local disk is temporary too, so download them to your own computer or Drive if you want to keep them. Containers expire after inactivity; do not use them as permanent storage.

In [ ]:
if RUN_DEMO and CLEANUP:
    for container_id in container_ids:
        client.containers.delete(container_id)
    if uploaded is not None:
        client.files.delete(uploaded.id)
    print("Deleted the demo's remote container and uploaded file; local results remain.")

## Try it yourself

Change one revenue value, rerun from the data cell, and check whether both the chart and verified totals change. Then ask for a monthly trend chart instead of a regional bar chart.

## Summary

- Code Interpreter runs model-written **Python in an OpenAI-hosted sandbox**.
- Attach files, ask for a concrete calculation, and inspect the tool call.
- Download generated files using their container citations.
- Check results against a known calculation and save files before the environment expires.

**Troubleshooting:** update the SDK if `containers` is missing; choose a supported model if the tool is rejected; check API billing if you receive a quota error. If execution stops before cleanup, run the cleanup cell once you have inspected and downloaded the results.

Sources: [Code Interpreter](https://developers.openai.com/api/docs/guides/tools-code-interpreter), [Container file downloads](https://developers.openai.com/api/reference/resources/containers/subresources/files/subresources/content/methods/retrieve).